# 🪢 Khipu ML — Structural Pattern Mining in Inka Khipus

**Author:** Maria Contreras | Universidad Peruana de Ciencias Aplicadas (UPC)  
**Dataset:** Open Khipu Repository (OKR) — DOI: 10.5281/zenodo.18025748  
**GitHub:** github.com/mcontrerasmalpar-pixel/khipu-ml  
**Paper (arXiv):** [link pendiente — agregar cuando esté aprobado]

---

## Abstract
This notebook presents a full ML pipeline applied to the Open Khipu Repository (OKR),
a public SQLite database of 619 Inka khipus. We extract 27 structural features per khipu
and apply unsupervised clustering (UMAP + HDBSCAN), supervised classification (XGBoost),
SHAP interpretability, and an independent computational validation of a published
decipherment claim (Medrano & Urton, 2018).

**Key results:**
- 3 structurally distinct clusters (silhouette = 0.769)
- XGBoost F1 = 0.86 for Inka Late Horizon style (tuned model)
- SHAP analysis reveals cord twist direction as the top discriminative feature
- Independent replication of the Santa Valley recto/verso moiety structure
  (KH0323–KH0328), matching Medrano & Urton (2018) without physical access
  to the objects
- Negative result: knot-type sequence n-grams add no provenance signal
  beyond aggregate features

This notebook is the computational companion to the paper *"Structural Pattern
Mining in Inka Khipus: Unsupervised Clustering, Provenance Classification, and
a Computational Validation of the Santa Valley Match"* (Contreras, 2026).

## 1. Setup & Exploratory Data Analysis
We connect to the OKR SQLite database and inspect the main tables:
- `khipu_main` — khipu-level metadata (provenance, museum, region)
- `cord` — per-cord attributes (length, twist, level, attachment)
- `knot` — per-knot attributes (type, direction, turns)
- `ascher_cord_color` — cord color encoding

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

DB_PATH = "/kaggle/input/datasets/macmaky/open-khipu-repository/khipu.db"
conn = sqlite3.connect(DB_PATH)

print("✓ Conexión exitosa")
print(f"  Tamaño del archivo: {os.path.getsize(DB_PATH) / 1e6:.1f} MB")

In [ ]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
# Ver todas las tablas disponibles
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn)
print(f"Tablas en khipu.db ({len(tables)} total):\n")
for t in tables['name']:
    count = pd.read_sql(f"SELECT COUNT(*) as n FROM {t}", conn)['n'][0]
    print(f"  {t:<35} {count:>7,} filas")

In [ ]:
khipu_main = pd.read_sql("SELECT * FROM khipu_main", conn)
print(f"Shape: {khipu_main.shape}")
display(khipu_main.head(3))
print("\nColumnas:", list(khipu_main.columns))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

completitud = (khipu_main.notna().mean() * 100).sort_values()
colors = ['#e74c3c' if v < 50 else '#f39c12' if v < 80 else '#2ecc71' 
          for v in completitud]

completitud.plot(kind='barh', ax=ax, color=colors)
ax.set_xlabel('% datos presentes')
ax.set_title('Completitud de columnas — khipu_main', fontweight='bold')
ax.axvline(80, color='gray', linestyle='--', alpha=0.5, label='80%')
ax.legend()
plt.tight_layout()
plt.savefig('/kaggle/working/completitud_khipu_main.png', dpi=150)
plt.show()
print(f"\nQuipus con procedencia conocida: {khipu_main['PROVENANCE'].notna().sum()} / {len(khipu_main)}")

In [ ]:
prov = khipu_main['PROVENANCE'].dropna().value_counts().head(20)

fig, ax = plt.subplots(figsize=(10, 6))
prov.plot(kind='barh', ax=ax, color='#8e44ad')
ax.set_xlabel('Número de quipus')
ax.set_title('Top 20 procedencias geográficas', fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/procedencia.png', dpi=150)
plt.show()

In [ ]:
cord_count = pd.read_sql("SELECT COUNT(*) as n FROM cord", conn)['n'][0]
cord_sample = pd.read_sql("SELECT * FROM cord LIMIT 3", conn)
print(f"Total de cordeles: {cord_count:,}")
display(cord_sample)

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Torsión S/Z
twist = pd.read_sql("""
    SELECT TWIST, COUNT(*) as n 
    FROM cord 
    WHERE TWIST IS NOT NULL
    GROUP BY TWIST ORDER BY n DESC
""", conn)
twist.plot(kind='bar', x='TWIST', y='n', ax=axes[0], color='#2980b9', legend=False)
axes[0].set_title('Distribución de torsión (TWIST)', fontweight='bold')
axes[0].set_xlabel('Torsión')
axes[0].set_ylabel('Número de cordeles')
axes[0].tick_params(axis='x', rotation=45)

# Nivel jerárquico
levels = pd.read_sql("""
    SELECT CORD_LEVEL, COUNT(*) as n 
    FROM cord 
    WHERE CORD_LEVEL IS NOT NULL
    GROUP BY CORD_LEVEL ORDER BY CORD_LEVEL
""", conn)
levels.plot(kind='bar', x='CORD_LEVEL', y='n', ax=axes[1], color='#27ae60', legend=False)
axes[1].set_title('Nivel jerárquico de cordeles', fontweight='bold')
axes[1].set_xlabel('Nivel')
axes[1].set_ylabel('Número de cordeles')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('/kaggle/working/cord_distribucion.png', dpi=150)
plt.show()

In [ ]:
knot_count = pd.read_sql("SELECT COUNT(*) as n FROM knot", conn)['n'][0]
print(f"Total de nudos: {knot_count:,}")

knot_types = pd.read_sql("""
    SELECT TYPE_CODE, COUNT(*) as n 
    FROM knot 
    WHERE TYPE_CODE IS NOT NULL
    GROUP BY TYPE_CODE ORDER BY n DESC
""", conn)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

knot_types.plot(kind='bar', x='TYPE_CODE', y='n', ax=axes[0],
                color='#e67e22', legend=False)
axes[0].set_title('Tipos de nudos (TYPE_CODE)', fontweight='bold')
axes[0].set_xlabel('Tipo')
axes[0].set_ylabel('Frecuencia')
axes[0].tick_params(axis='x', rotation=45)

# Dirección S/Z de nudos
direction = pd.read_sql("""
    SELECT DIRECTION, COUNT(*) as n
    FROM knot
    WHERE DIRECTION IS NOT NULL
    GROUP BY DIRECTION ORDER BY n DESC
""", conn)
direction.plot(kind='bar', x='DIRECTION', y='n', ax=axes[1],
               color='#8e44ad', legend=False)
axes[1].set_title('Dirección de nudos (S/Z)', fontweight='bold')
axes[1].set_xlabel('Dirección')
axes[1].set_ylabel('Frecuencia')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('/kaggle/working/knot_types.png', dpi=150)
plt.show()

In [ ]:
print("=" * 50)
print("RESUMEN DEL CORPUS OKR")
print("=" * 50)

quipus_total = pd.read_sql("SELECT COUNT(*) as n FROM khipu_main", conn)['n'][0]
quipus_con_prov = khipu_main['PROVENANCE'].notna().sum()
quipus_con_region = khipu_main['REGION'].notna().sum()
cords_total = pd.read_sql("SELECT COUNT(*) as n FROM cord", conn)['n'][0]
knots_total = pd.read_sql("SELECT COUNT(*) as n FROM knot", conn)['n'][0]

print(f"  Quipus totales:              {quipus_total:>6,}")
print(f"  Quipus con PROVENANCE:       {quipus_con_prov:>6,} ({quipus_con_prov/quipus_total*100:.1f}%)")
print(f"  Quipus con REGION:           {quipus_con_region:>6,} ({quipus_con_region/quipus_total*100:.1f}%)")
print(f"  Cordeles totales:            {cords_total:>6,}")
print(f"  Nudos totales:               {knots_total:>6,}")
print(f"\n  → Usables para clustering:   ~{quipus_total - 20:>4,}")
print(f"  → Usables para clasificación: ~{max(quipus_con_prov, quipus_con_region):>4,}")
print("=" * 50)

conn.close()
print("\n✓ Conexión cerrada")

---
## 2. Feature Engineering
We aggregate cord and knot attributes into a 27-dimensional feature vector per khipu:

| Feature group | Features |
|---|---|
| Cord structure | n_cords, mean_length, std_length, max_cord_level |
| Hierarchy | n_subsidiary, ratio_subsidiary |
| Twist | twist_S, twist_Z, ratio_SZ |
| Knots | n_knots, mean_turns, knot_dir_S, knot_dir_Z, ratio_knot_SZ |
| Knot types | knot_S, knot_L, knot_E, knot_LL, knot_BL (one-hot counts) |
| Color | n_unique_colors, color_entropy, has_multicolor |

In [ ]:
import sqlite3
import pandas as pd
import numpy as np

DB_PATH = "/kaggle/input/datasets/macmaky/open-khipu-repository/khipu.db"
conn = sqlite3.connect(DB_PATH)

khipu_main = pd.read_sql("SELECT KHIPU_ID, PROVENANCE, REGION, MUSEUM_NAME FROM khipu_main", conn)
cord = pd.read_sql("SELECT KHIPU_ID, CORD_ID, CORD_LEVEL, CORD_LENGTH, TWIST, ATTACHMENT_TYPE, FIBER FROM cord", conn)
knot = pd.read_sql("SELECT CORD_ID, TYPE_CODE, DIRECTION, NUM_TURNS FROM knot", conn)
colors = pd.read_sql("SELECT KHIPU_ID, CORD_ID, COLOR_CD_1, COLOR_CD_2 FROM ascher_cord_color", conn)

print(f"khipu_main: {khipu_main.shape}")
print(f"cord:       {cord.shape}")
print(f"knot:       {knot.shape}")
print(f"colors:     {colors.shape}")

In [ ]:
def cord_features(cord_df):
    feats = cord_df.groupby('KHIPU_ID').agg(
        n_cords            = ('CORD_ID', 'count'),
        mean_length        = ('CORD_LENGTH', 'mean'),
        std_length         = ('CORD_LENGTH', 'std'),
        max_cord_level     = ('CORD_LEVEL', 'max'),
        n_subsidiary       = ('CORD_LEVEL', lambda x: (x > 1).sum()),
        ratio_subsidiary   = ('CORD_LEVEL', lambda x: (x > 1).mean()),
        twist_S            = ('TWIST', lambda x: (x == 'S').sum()),
        twist_Z            = ('TWIST', lambda x: (x == 'Z').sum()),
    ).reset_index()

    feats['ratio_SZ'] = feats['twist_S'] / (feats['twist_S'] + feats['twist_Z'] + 1e-9)
    feats['std_length'] = feats['std_length'].fillna(0)
    return feats

cord_feats = cord_features(cord)
print(f"Features de cordeles: {cord_feats.shape}")
display(cord_feats.head(3))

In [ ]:
def knot_features(knot_df, cord_df):
    # join cord → knot para tener KHIPU_ID
    merged = knot_df.merge(cord_df[['CORD_ID', 'KHIPU_ID']], on='CORD_ID', how='left')

    # distribución de tipos de nudo
    type_dummies = pd.get_dummies(merged['TYPE_CODE'], prefix='knot')
    merged = pd.concat([merged[['KHIPU_ID']], type_dummies], axis=1)
    knot_type_feats = merged.groupby('KHIPU_ID').sum().reset_index()

    # métricas generales
    knot_general = knot_df.merge(cord_df[['CORD_ID', 'KHIPU_ID']], on='CORD_ID', how='left')
    knot_general = knot_general.groupby('KHIPU_ID').agg(
        n_knots        = ('TYPE_CODE', 'count'),
        mean_turns     = ('NUM_TURNS', 'mean'),
        knot_dir_S     = ('DIRECTION', lambda x: (x == 'S').sum()),
        knot_dir_Z     = ('DIRECTION', lambda x: (x == 'Z').sum()),
    ).reset_index()
    knot_general['ratio_knot_SZ'] = (
        knot_general['knot_dir_S'] / 
        (knot_general['knot_dir_S'] + knot_general['knot_dir_Z'] + 1e-9)
    )

    feats = knot_general.merge(knot_type_feats, on='KHIPU_ID', how='left')
    return feats

knot_feats = knot_features(knot, cord)
print(f"Features de nudos: {knot_feats.shape}")
display(knot_feats.head(3))

In [ ]:
from scipy.stats import entropy

def color_features(colors_df):
    # entropía de color por quipu
    def color_entropy(series):
        counts = series.dropna().value_counts(normalize=True)
        return entropy(counts) if len(counts) > 1 else 0.0

    feats = colors_df.groupby('KHIPU_ID').agg(
        n_unique_colors  = ('COLOR_CD_1', 'nunique'),
        color_entropy    = ('COLOR_CD_1', color_entropy),
        has_multicolor   = ('COLOR_CD_2', lambda x: x.notna().any().astype(int)),
    ).reset_index()
    return feats

color_feats = color_features(colors)
print(f"Features de colores: {color_feats.shape}")
display(color_feats.head(3))

In [ ]:
feature_matrix = khipu_main.copy()
feature_matrix = feature_matrix.merge(cord_feats,  on='KHIPU_ID', how='left')
feature_matrix = feature_matrix.merge(knot_feats,  on='KHIPU_ID', how='left')
feature_matrix = feature_matrix.merge(color_feats, on='KHIPU_ID', how='left')

# columnas numéricas para ML
meta_cols = ['KHIPU_ID', 'PROVENANCE', 'REGION', 'MUSEUM_NAME']
feature_cols = [c for c in feature_matrix.columns if c not in meta_cols]

# imputar NaN con 0
feature_matrix[feature_cols] = feature_matrix[feature_cols].fillna(0)

print(f"Feature matrix final: {feature_matrix.shape}")
print(f"Features numéricas:   {len(feature_cols)}")
print(f"\nFeatures disponibles:\n{feature_cols}")

# guardar para los siguientes pasos
feature_matrix.to_csv('/kaggle/working/feature_matrix.csv', index=False)
print("\n✓ Guardado en /kaggle/working/feature_matrix.csv")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

X = feature_matrix[feature_cols]

plt.figure(figsize=(14, 10))
corr = X.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0,
            linewidths=0.3, annot=False, fmt='.2f')
plt.title('Correlación entre features — matriz de quipus', fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/correlacion_features.png', dpi=150)
plt.show()

conn.close()
print("✓ Feature engineering completo")


---
## 3a. Unsupervised Clustering — UMAP + HDBSCAN
We apply UMAP for dimensionality reduction followed by HDBSCAN density-based clustering.
No prior knowledge of the number of clusters is assumed.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from umap import UMAP
import hdbscan
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

feature_matrix = pd.read_csv('/kaggle/working/feature_matrix.csv')

meta_cols = ['KHIPU_ID', 'PROVENANCE', 'REGION', 'MUSEUM_NAME']
feature_cols = [c for c in feature_matrix.columns if c not in meta_cols]

X = feature_matrix[feature_cols].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Matrix para clustering: {X_scaled.shape}")

In [ ]:
umap_model = UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric='euclidean',
    random_state=42
)

X_umap = umap_model.fit_transform(X_scaled)
print(f"UMAP output: {X_umap.shape}")

plt.figure(figsize=(8, 6))
plt.scatter(X_umap[:, 0], X_umap[:, 1], s=18, alpha=0.7, color='#8e44ad')
plt.title('UMAP — 619 quipus (sin etiquetas)', fontweight='bold')
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.tight_layout()
plt.savefig('/kaggle/working/umap_raw.png', dpi=150)
plt.show()

In [ ]:
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=10,
    min_samples=5,
    metric='euclidean'
)

labels = clusterer.fit_predict(X_umap)
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = (labels == -1).sum()

print(f"Clusters encontrados: {n_clusters}")
print(f"Puntos ruido (-1):    {n_noise} ({n_noise/len(labels)*100:.1f}%)")

# silhouette solo sobre puntos asignados
mask = labels != -1
if mask.sum() > 1:
    sil = silhouette_score(X_umap[mask], labels[mask])
    print(f"Silhouette score:     {sil:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: clusters HDBSCAN
palette = sns.color_palette('tab10', n_clusters)
color_map = {-1: (0.7, 0.7, 0.7)}  # gris para ruido
for i, c in enumerate(sorted(set(labels))):
    if c != -1:
        color_map[c] = palette[i % len(palette)]

colors_plot = [color_map[l] for l in labels]
axes[0].scatter(X_umap[:, 0], X_umap[:, 1], c=colors_plot, s=18, alpha=0.8)
axes[0].set_title(f'HDBSCAN — {n_clusters} clusters', fontweight='bold')
axes[0].set_xlabel('UMAP 1')
axes[0].set_ylabel('UMAP 2')

# Plot 2: coloreado por REGION
regions = feature_matrix['REGION'].fillna('Unknown')
unique_regions = regions.unique()
region_palette = sns.color_palette('Set2', len(unique_regions))
region_color_map = {r: region_palette[i] for i, r in enumerate(unique_regions)}
region_colors = [region_color_map[r] for r in regions]

axes[1].scatter(X_umap[:, 0], X_umap[:, 1], c=region_colors, s=18, alpha=0.8)
axes[1].set_title('Coloreado por REGION geográfica', fontweight='bold')
axes[1].set_xlabel('UMAP 1')
axes[1].set_ylabel('UMAP 2')

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=region_palette[i], label=r) 
                   for i, r in enumerate(unique_regions)]
axes[1].legend(handles=legend_elements, fontsize=7, 
               loc='upper right', ncol=2)

plt.tight_layout()
plt.savefig('/kaggle/working/clustering_resultado.png', dpi=150)
plt.show()

In [ ]:
feature_matrix['cluster'] = labels

print("Distribución de quipus por cluster:\n")
for c in sorted(feature_matrix['cluster'].unique()):
    subset = feature_matrix[feature_matrix['cluster'] == c]
    top_region = subset['REGION'].value_counts().index[0] if subset['REGION'].notna().any() else 'N/A'
    print(f"  Cluster {c:>2}: {len(subset):>3} quipus | región dominante: {top_region}")

In [ ]:
# Perfil detallado por cluster
print("═" * 60)
for c in [0, 1, 2]:
    subset = feature_matrix[feature_matrix['cluster'] == c]
    print(f"\nCLUSTER {c} — {len(subset)} quipus")
    print(f"  Top 3 regiones:")
    print(subset['REGION'].value_counts().head(3).to_string())
    print(f"\n  Medias de features clave:")
    for feat in ['n_cords', 'mean_length', 'ratio_subsidiary', 
                 'ratio_SZ', 'n_knots', 'color_entropy']:
        val = subset[feat].mean()
        print(f"    {feat:<22} {val:.3f}")
print("\n═" * 60)

# Top museos en cluster 2
print("\nCluster 2 — Top museos:")
print(feature_matrix[feature_matrix['cluster']==2]['MUSEUM_NAME'].value_counts().head(5))

---
## 3b. Supervised Classification — XGBoost
We train a gradient boosting classifier to predict geographic region of origin
from structural features, using 5-fold stratified cross-validation.

Labels: REGION column from khipu_main (84.3% coverage, 522 khipus).
Classes with fewer than 10 samples grouped as 'Other'.

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# usar REGION como etiqueta (84.3% cobertura)
df_sup = feature_matrix[feature_matrix['REGION'].notna()].copy()

# agrupar regiones con menos de 10 quipus en 'Other'
region_counts = df_sup['REGION'].value_counts()
df_sup['REGION_LABEL'] = df_sup['REGION'].apply(
    lambda x: x if region_counts[x] >= 10 else 'Other'
)

print("Distribución de clases:")
print(df_sup['REGION_LABEL'].value_counts().to_string())
print(f"\nTotal muestras: {len(df_sup)}")
print(f"Clases: {df_sup['REGION_LABEL'].nunique()}")

In [ ]:
le = LabelEncoder()
y = le.fit_transform(df_sup['REGION_LABEL'])
X_sup = df_sup[feature_cols].values

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(xgb, X_sup, y, cv=cv, scoring='f1_weighted')

print(f"F1 weighted (5-fold CV):")
print(f"  {scores.round(3)}")
print(f"  Media: {scores.mean():.3f} ± {scores.std():.3f}")

In [ ]:
xgb.fit(X_sup, y)

importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': xgb.feature_importances_
}).sort_values('importance', ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(9, 6))
importance_df.plot(kind='barh', x='feature', y='importance', 
                   ax=ax, color='#e74c3c', legend=False)
ax.set_title('Top 15 features — importancia XGBoost', fontweight='bold')
ax.set_xlabel('Importancia')
plt.tight_layout()
plt.savefig('/kaggle/working/feature_importance.png', dpi=150)
plt.show()

In [ ]:
from sklearn.model_selection import cross_val_predict

y_pred = cross_val_predict(xgb, X_sup, y, cv=cv)

fig, ax = plt.subplots(figsize=(10, 8))
cm = confusion_matrix(y, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_,
            yticklabels=le.classes_, ax=ax)
ax.set_title('Confusion Matrix — clasificación por región', fontweight='bold')
ax.set_xlabel('Predicho')
ax.set_ylabel('Real')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrix.png', dpi=150)
plt.show()

print("\nReporte de clasificación:")
print(classification_report(y, y_pred, target_names=le.classes_))

---
## 4. Interpretation & Visualization
We combine clustering and classification results to build a multi-dimensional 
view of the khipu corpus, coloring the UMAP embedding by cluster, 
geographic region, and color entropy.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: clusters HDBSCAN
scatter1 = axes[0].scatter(X_umap[:, 0], X_umap[:, 1],
                            c=feature_matrix['cluster'],
                            cmap='Set1', s=18, alpha=0.8)
axes[0].set_title('Clusters HDBSCAN', fontweight='bold')
axes[0].set_xlabel('UMAP 1'); axes[0].set_ylabel('UMAP 2')
plt.colorbar(scatter1, ax=axes[0], label='Cluster')

# Plot 2: región geográfica
region_encoded = LabelEncoder().fit_transform(
    feature_matrix['REGION'].fillna('Unknown'))
scatter2 = axes[1].scatter(X_umap[:, 0], X_umap[:, 1],
                            c=region_encoded,
                            cmap='tab20', s=18, alpha=0.8)
axes[1].set_title('Región geográfica', fontweight='bold')
axes[1].set_xlabel('UMAP 1'); axes[1].set_ylabel('UMAP 2')

# Plot 3: color_entropy
scatter3 = axes[2].scatter(X_umap[:, 0], X_umap[:, 1],
                            c=feature_matrix['color_entropy'],
                            cmap='YlOrRd', s=18, alpha=0.8)
axes[2].set_title('Entropía de color', fontweight='bold')
axes[2].set_xlabel('UMAP 1'); axes[2].set_ylabel('UMAP 2')
plt.colorbar(scatter3, ax=axes[2], label='Entropía')

plt.suptitle('Quipus OKR — análisis multidimensional', 
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/kaggle/working/interpretacion_final.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print("═" * 60)
print("HALLAZGOS PRINCIPALES")
print("═" * 60)

print("""
1. CLUSTERING (UMAP + HDBSCAN)
   • 3 clusters estructuralmente distintos, silhouette = 0.769
   • Cluster 0 (17):  estilo imperial Inka tardío
   • Cluster 1 (442): costa central del Perú — estilo mayoritario
   • Cluster 2 (160): colecciones europeas + excavaciones de campo

2. CLASIFICACIÓN (XGBoost)
   • F1 weighted = 0.46 sobre 7 regiones (baseline)
   • Mejor clase: "Inka, Late Horizon" F1 = 0.80
   • Features más predictivas: ver feature_importance.png

3. IMPLICACIONES ARQUEOLÓGICAS
   • Las convenciones de construcción Inka imperial son
     altamente estandarizadas y ML-detectables
   • La procedencia geográfica está parcialmente codificada
     en la estructura física del quipu
   • El Cluster 2 sugiere patrones de recolección colonial
     que afectan la distribución del corpus
""")
print("═" * 60)

# guardar feature matrix con clusters
feature_matrix.to_csv('/kaggle/working/feature_matrix_clusters.csv', index=False)
print("✓ Guardado feature_matrix_clusters.csv")
print("✓ Pipeline completo")

---
## 5. Interpretability — SHAP Values
We use SHAP TreeExplainer to identify which structural features drive
classification of Inka Late Horizon style (F1 = 0.80).

In [ ]:
import shap
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

# recargar datos
feature_matrix = pd.read_csv('/kaggle/working/feature_matrix.csv')

meta_cols = ['KHIPU_ID', 'PROVENANCE', 'REGION', 'MUSEUM_NAME']
feature_cols = [c for c in feature_matrix.columns if c not in meta_cols]

df_sup = feature_matrix[feature_matrix['REGION'].notna()].copy()
region_counts = df_sup['REGION'].value_counts()
df_sup['REGION_LABEL'] = df_sup['REGION'].apply(
    lambda x: x if region_counts[x] >= 10 else 'Other'
)

le = LabelEncoder()
y = le.fit_transform(df_sup['REGION_LABEL'])
X_sup = df_sup[feature_cols].values

# entrenar modelo final
xgb = XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='mlogloss', random_state=42
)
xgb.fit(X_sup, y)
print("✓ Modelo entrenado")
print(f"  Clases: {list(le.classes_)}")

In [ ]:
explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_sup)

print(f"Shape SHAP values: {np.array(shap_values).shape}")
print("✓ SHAP values calculados")

In [ ]:
# shape (135, 27, 7) → mean sobre muestras → (27, 7) → mean sobre clases → (27,)
shap_global = np.abs(shap_values).mean(axis=0).mean(axis=1)

print(f"shap_global shape: {shap_global.shape}")  # debe ser (27,)

importance_shap = pd.DataFrame({
    'feature': feature_cols,
    'shap_importance': shap_global
}).sort_values('shap_importance', ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(9, 6))
importance_shap.plot(kind='barh', x='feature', y='shap_importance',
                     ax=ax, color='#8e44ad', legend=False)
ax.set_title('Top 15 features — importancia SHAP global', fontweight='bold')
ax.set_xlabel('Mean |SHAP value|')
plt.tight_layout()
plt.savefig('/kaggle/working/shap_global.png', dpi=150)
plt.show()

In [ ]:
print(list(le.classes_))

In [ ]:
inka_idx = 1  # "Inka, Late Horizon"
print(f"Clase: {le.classes_[inka_idx]}")

shap.summary_plot(
    shap_values[:, :, inka_idx],  # shape (135, 27)
    X_sup,
    feature_names=feature_cols,
    plot_type='dot',
    max_display=15,
    show=False
)
plt.title('SHAP — "Inka, Late Horizon" (F1=0.80)', fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/shap_inka.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Conclusions & Key Findings

In [ ]:

print("""
╔══════════════════════════════════════════════════════════════╗
║         KHIPU ML — FINAL RESULTS                            ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  CLUSTERING (UMAP + HDBSCAN)                                 ║
║  · 619 khipus → 3 structurally distinct clusters             ║
║  · Silhouette = 0.769 (excellent separation)                 ║
║  · Cluster 0: Inka Late Horizon imperial style (17 khipus)   ║
║  · Cluster 1: Central Coast, Peru (442 khipus)               ║
║  · Cluster 2: European collections s.XIX (160 khipus)        ║
║                                                              ║
║  CLASSIFICATION (XGBoost)                                    ║
║  · F1 weighted = 0.46 (baseline, 7 classes)                  ║
║  · F1 = 0.80 for "Inka, Late Horizon"                        ║
║                                                              ║
║  INTERPRETABILITY (SHAP)                                     ║
║  · Feature #1: knot_dir_Z — standardized Z-direction         ║
║  · Feature #2: mean_length — shorter cord lengths            ║
║  · Feature #3: n_knots — higher knot density                 ║
║  · Feature #4: std_length — high uniformity                  ║
║  · Feature #5: color_entropy — restricted color palette      ║
║                                                              ║
║  ARCHAEOLOGICAL IMPLICATION                                  ║
║  "Inka imperial khipus show highly standardized              ║
║   Z-direction knotting, shorter uniform cord lengths,        ║
║   and restricted color palettes — all ML-detectable          ║
║   with F1=0.80."                                             ║
║                                                              ║
║  Dataset: Open Khipu Repository (OKR)                        ║
║  DOI: 10.5281/zenodo.18025748                                ║
║  Code: github.com/mcontrerasmalpar-pixel/khipu-ml            ║
╚══════════════════════════════════════════════════════════════╝
""")

In [ ]:
import os

outputs = os.listdir('/kaggle/working')
print("Outputs guardados:")
for f in sorted(outputs):
    size = os.path.getsize(f'/kaggle/working/{f}') / 1024
    print(f"  {f:<45} {size:.1f} KB")

---
## 7. Deep Analysis — Cluster 2: Colonial Collection Bias
Investigating why European museum collections form a structurally distinct cluster.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DB_PATH = "/kaggle/input/datasets/macmaky/open-khipu-repository/khipu.db"
conn = sqlite3.connect(DB_PATH)

feature_matrix = pd.read_csv('/kaggle/working/feature_matrix_clusters.csv')
print(f"Shape: {feature_matrix.shape}")
print(f"Clusters: {feature_matrix['cluster'].value_counts().to_dict()}")

In [ ]:
meta_cols = ['KHIPU_ID', 'PROVENANCE', 'REGION', 'MUSEUM_NAME', 'cluster']
feature_cols = [c for c in feature_matrix.columns if c not in meta_cols]

# comparar medias de features por cluster
profile = feature_matrix.groupby('cluster')[feature_cols].mean().T
profile.columns = ['Cluster 0\n(Inka imperial)', 
                   'Cluster 1\n(Central Coast)', 
                   'Cluster 2\n(European collections)']

# normalizar para visualización
profile_norm = (profile - profile.min()) / (profile.max() - profile.min() + 1e-9)

fig, ax = plt.subplots(figsize=(12, 8))
profile_norm.plot(kind='bar', ax=ax, 
                  color=['#e74c3c', '#3498db', '#2ecc71'],
                  width=0.75, alpha=0.85)
ax.set_title('Feature profile by cluster (normalized)', fontweight='bold', fontsize=13)
ax.set_xlabel('Feature')
ax.set_ylabel('Normalized mean value')
ax.legend(loc='upper right')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('/kaggle/working/cluster2_profile.png', dpi=150)
plt.show()

In [ ]:
from scipy import stats

cluster2 = feature_matrix[feature_matrix['cluster'] == 2]
cluster1 = feature_matrix[feature_matrix['cluster'] == 1]

results = []
for feat in feature_cols:
    t_stat, p_val = stats.ttest_ind(
        cluster2[feat].dropna(),
        cluster1[feat].dropna()
    )
    mean_c2 = cluster2[feat].mean()
    mean_c1 = cluster1[feat].mean()
    results.append({
        'feature': feat,
        'mean_cluster2': mean_c2,
        'mean_cluster1': mean_c1,
        'difference': mean_c2 - mean_c1,
        'p_value': p_val,
        'significant': p_val < 0.05
    })

results_df = pd.DataFrame(results).sort_values('p_value')
sig = results_df[results_df['significant']].head(10)

print("Features que distinguen significativamente Cluster 2 de Cluster 1:")
print(sig[['feature', 'mean_cluster2', 'mean_cluster1', 'difference', 'p_value']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top museos Cluster 2
c2_museums = feature_matrix[feature_matrix['cluster'] == 2]['MUSEUM_NAME'].value_counts().head(10)
c2_museums.plot(kind='barh', ax=axes[0], color='#2ecc71')
axes[0].set_title('Top museums — Cluster 2\n(European collections)', fontweight='bold')
axes[0].set_xlabel('Number of khipus')

# Top museos Cluster 1
c1_museums = feature_matrix[feature_matrix['cluster'] == 1]['MUSEUM_NAME'].value_counts().head(10)
c1_museums.plot(kind='barh', ax=axes[1], color='#3498db')
axes[1].set_title('Top museums — Cluster 1\n(Central Coast)', fontweight='bold')
axes[1].set_xlabel('Number of khipus')

plt.tight_layout()
plt.savefig('/kaggle/working/cluster2_museums.png', dpi=150)
plt.show()

In [ ]:
cord = pd.read_sql("""
    SELECT c.KHIPU_ID, c.CORD_LENGTH, c.TWIST, c.CORD_LEVEL
    FROM cord c
""", conn)

cord_clusters = cord.merge(
    feature_matrix[['KHIPU_ID', 'cluster']], 
    on='KHIPU_ID', how='left'
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribución de longitud por cluster
for c, color, label in zip([0, 1, 2], 
                             ['#e74c3c', '#3498db', '#2ecc71'],
                             ['Cluster 0 (Inka)', 'Cluster 1 (Coast)', 'Cluster 2 (European)']):
    data = cord_clusters[cord_clusters['cluster'] == c]['CORD_LENGTH'].dropna()
    axes[0].hist(data, bins=50, alpha=0.6, color=color, label=label, density=True)

axes[0].set_title('Cord length distribution by cluster', fontweight='bold')
axes[0].set_xlabel('Cord length')
axes[0].set_ylabel('Density')
axes[0].legend()
axes[0].set_xlim(0, 200)

# Twist S/Z por cluster
twist_summary = cord_clusters.groupby(['cluster', 'TWIST']).size().unstack(fill_value=0)
twist_pct = twist_summary.div(twist_summary.sum(axis=1), axis=0) * 100
twist_pct.plot(kind='bar', ax=axes[1], colormap='Set2')
axes[1].set_title('Twist direction (S/Z) by cluster (%)', fontweight='bold')
axes[1].set_xlabel('Cluster')
axes[1].set_ylabel('Percentage (%)')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('/kaggle/working/cluster2_physical.png', dpi=150)
plt.show()

conn.close()

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════╗
║         CLUSTER 2 — COLONIAL COLLECTION BIAS                ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  COMPOSITION                                                 ║
║  · 160 khipus from European & North American museums         ║
║  · Dominated by 19th-century colonial acquisitions           ║
║  · Museum für Völkerkunde Berlin: largest single source      ║
║                                                              ║
║  STRUCTURAL DIFFERENCES vs Cluster 1                         ║
║  · See t-test results above for significant features         ║
║                                                              ║
║  ARCHAEOLOGICAL IMPLICATION                                  ║
║  · Collection method and institutional origin leave a        ║
║    detectable structural signature in the corpus             ║
║  · OKR is not a neutral sample of Inka khipus                ║
║  · Decipherment efforts must account for this bias           ║
║                                                              ║
║  NEXT STEPS                                                  ║
║  · Investigate recording scholar as confound                 ║
║  · Cross-reference with excavation context data              ║
║  · Test if bias affects summation pattern analysis           ║
╚══════════════════════════════════════════════════════════════╝
""")

In [ ]:
# Investigar torsión U en Cluster 2
print("Distribución TWIST por cluster (valores exactos):")
twist_detail = cord_clusters.groupby(['cluster', 'TWIST']).size().unstack(fill_value=0)
twist_pct = twist_detail.div(twist_detail.sum(axis=1), axis=0) * 100
print(twist_pct.round(1).to_string())

print("\nValores únicos de TWIST en el dataset:")
print(cord_clusters['TWIST'].value_counts().head(10))

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════╗
║      CLUSTER 2 — HALLAZGO FINAL ACTUALIZADO                 ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  TWIST S/U/Z REVISADO                                        ║
║  · "U" = unspecified (dato no registrado, no categoría)      ║
║  · Cluster 0 Inka:    85.3% S — torsión S estandarizada     ║
║  · Cluster 1 Costa:   91.0% U — datos mayormente faltantes  ║
║  · Cluster 2 Europeo: 69.7% U + 26.2% Z — mixto            ║
║                                                              ║
║  CORRECCIÓN AL HALLAZGO SHAP                                 ║
║  · La torsión S es el marcador Inka imperial                 ║
║  · knot_dir_Z predice porque su AUSENCIA = estilo Inka       ║
║  · Los quipus imperiales usan S en cordeles, Z en nudos      ║
║                                                              ║
║  CLUSTER 2 — INTERPRETACIÓN FINAL                            ║
║  · Quipus con datos de torsión incompletos (U)               ║
║  · Colectados por instituciones con protocolos de            ║
║    registro distintos al estándar Ascher/Urton               ║
║  · El "bias" es parcialmente un artefacto de                 ║
║    inconsistencia metodológica en el registro                ║
║                                                              ║
║  IMPLICACIÓN PARA EL PAPER                                   ║
║  · El Cluster 2 refleja TANTO bias colonial COMO             ║
║    heterogeneidad en los protocolos de registro              ║
║  · Ambos factores son consecuencia del colonialismo          ║
╚══════════════════════════════════════════════════════════════╝
""")

---
## 8. XGBoost Tuning — Improving Minority Classes
Hyperparameter optimization to improve F1 for underrepresented regions.

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
import pandas as pd
import numpy as np

feature_matrix = pd.read_csv('/kaggle/working/feature_matrix_clusters.csv')

meta_cols = ['KHIPU_ID', 'PROVENANCE', 'REGION', 'MUSEUM_NAME', 'cluster']
feature_cols = [c for c in feature_matrix.columns if c not in meta_cols]

df_sup = feature_matrix[feature_matrix['REGION'].notna()].copy()
region_counts = df_sup['REGION'].value_counts()
df_sup['REGION_LABEL'] = df_sup['REGION'].apply(
    lambda x: x if region_counts[x] >= 10 else 'Other'
)

le = LabelEncoder()
y = le.fit_transform(df_sup['REGION_LABEL'])
X_sup = df_sup[feature_cols].values

print(f"Samples: {len(y)}")
print(f"Classes: {list(le.classes_)}")
print(f"\nClass distribution:")
for i, c in enumerate(le.classes_):
    print(f"  {c}: {(y==i).sum()}")

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

# compute class weights para clases minoritarias
class_weights = compute_class_weight('balanced', classes=np.unique(y), y=y)
weight_dict = dict(zip(np.unique(y), class_weights))
print("Class weights:")
for i, c in enumerate(le.classes_):
    print(f"  {c}: {class_weights[i]:.3f}")

# baseline con class weights
xgb_baseline = XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='mlogloss', random_state=42
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
from sklearn.model_selection import cross_val_score
scores_baseline = cross_val_score(xgb_baseline, X_sup, y, cv=cv, scoring='f1_weighted')
print(f"\nBaseline F1 weighted: {scores_baseline.mean():.3f} ± {scores_baseline.std():.3f}")

In [ ]:
param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.6, 0.7, 0.8, 0.9],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.2, 0.5],
    'reg_alpha': [0, 0.1, 0.5, 1.0],
    'reg_lambda': [1.0, 1.5, 2.0]
}

xgb_search = XGBClassifier(
    eval_metric='mlogloss',
    random_state=42
)

search = RandomizedSearchCV(
    xgb_search,
    param_distributions=param_dist,
    n_iter=50,
    scoring='f1_weighted',
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

search.fit(X_sup, y)
print(f"\nBest F1 weighted: {search.best_score_:.3f}")
print(f"Best params: {search.best_params_}")

In [ ]:
from sklearn.model_selection import cross_val_predict

xgb_tuned = search.best_estimator_
scores_tuned = cross_val_score(xgb_tuned, X_sup, y, cv=cv, scoring='f1_weighted')

print("=" * 50)
print(f"Baseline F1:  {scores_baseline.mean():.3f} ± {scores_baseline.std():.3f}")
print(f"Tuned F1:     {scores_tuned.mean():.3f} ± {scores_tuned.std():.3f}")
print(f"Improvement:  +{(scores_tuned.mean() - scores_baseline.mean()):.3f}")
print("=" * 50)

y_pred_tuned = cross_val_predict(xgb_tuned, X_sup, y, cv=cv)
print("\nClassification report (tuned):")
print(classification_report(y, y_pred_tuned, target_names=le.classes_))

---
## 9. Investigating Central Coast / South Coast Confusion

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y, y_pred_tuned)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
ax.set_title('Confusion Matrix — Tuned XGBoost', fontweight='bold')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrix_tuned.png', dpi=150)
plt.show()

In [ ]:
from scipy import stats

central = df_sup[df_sup['REGION_LABEL'] == '"Central Coast, Peru"']
south = df_sup[df_sup['REGION_LABEL'] == '"South Coast, Peru"']

results = []
for feat in feature_cols:
    t_stat, p_val = stats.ttest_ind(central[feat].dropna(), south[feat].dropna())
    results.append({
        'feature': feat,
        'mean_central': central[feat].mean(),
        'mean_south': south[feat].mean(),
        'p_value': p_val
    })

results_df = pd.DataFrame(results).sort_values('p_value')
print("Features comparando Central Coast vs South Coast:")
print(results_df.head(10).to_string(index=False))
print(f"\nFeatures con p<0.05: {(results_df['p_value']<0.05).sum()} de {len(feature_cols)}")

---
## 10. Sequence Modeling — N-grams of Knot Types
Exploring whether ordered knot sequences carry signal beyond aggregate counts.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np

DB_PATH = "/kaggle/input/datasets/macmaky/open-khipu-repository/khipu.db"
conn = sqlite3.connect(DB_PATH)

knot_seq = pd.read_sql("""
    SELECT k.CORD_ID, c.KHIPU_ID, k.TYPE_CODE, k.KNOT_ORDINAL
    FROM knot k
    JOIN cord c ON k.CORD_ID = c.CORD_ID
    WHERE k.TYPE_CODE IS NOT NULL AND k.KNOT_ORDINAL IS NOT NULL
    ORDER BY c.KHIPU_ID, k.CORD_ID, k.KNOT_ORDINAL
""", conn)

# secuencia por khipu (concatenando cordeles en orden)
sequences = knot_seq.groupby('KHIPU_ID')['TYPE_CODE'].apply(list).reset_index()
sequences['seq_len'] = sequences['TYPE_CODE'].apply(len)

print(f"Khipus con secuencia: {len(sequences)}")
print(f"Longitud: min={sequences['seq_len'].min()}, median={sequences['seq_len'].median():.0f}, max={sequences['seq_len'].max()}")

In [ ]:
from collections import Counter

def get_ngrams(seq, n):
    return ['-'.join(seq[i:i+n]) for i in range(len(seq)-n+1)]

sequences['bigrams'] = sequences['TYPE_CODE'].apply(lambda s: get_ngrams(s, 2))
sequences['trigrams'] = sequences['TYPE_CODE'].apply(lambda s: get_ngrams(s, 3))

# top bigramas globales
all_bigrams = Counter([bg for seq in sequences['bigrams'] for bg in seq])
print("Top 15 bigramas más comunes en el corpus:")
for bg, count in all_bigrams.most_common(15):
    print(f"  {bg:<10} {count}")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# convertir secuencias a texto tipo "S-S-L S-L-E ..."
sequences['bigram_text'] = sequences['bigrams'].apply(lambda x: ' '.join(x))

tfidf = TfidfVectorizer(max_features=50, min_df=5)
X_ngram = tfidf.fit_transform(sequences['bigram_text']).toarray()

ngram_cols = [f"ngram_{f}" for f in tfidf.get_feature_names_out()]
ngram_df = pd.DataFrame(X_ngram, columns=ngram_cols)
ngram_df['KHIPU_ID'] = sequences['KHIPU_ID'].values

print(f"N-gram features: {ngram_df.shape}")
print(f"\nTop features: {ngram_cols[:10]}")

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder

feature_matrix = pd.read_csv('/kaggle/working/feature_matrix_clusters.csv')
meta_cols = ['KHIPU_ID', 'PROVENANCE', 'REGION', 'MUSEUM_NAME', 'cluster']
base_feature_cols = [c for c in feature_matrix.columns if c not in meta_cols]

# combinar features base + n-gramas
combined = feature_matrix.merge(ngram_df, on='KHIPU_ID', how='inner')
df_sup = combined[combined['REGION'].notna()].copy()

region_counts = df_sup['REGION'].value_counts()
df_sup['REGION_LABEL'] = df_sup['REGION'].apply(lambda x: x if region_counts[x] >= 10 else 'Other')

le = LabelEncoder()
y = le.fit_transform(df_sup['REGION_LABEL'])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
xgb = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                    subsample=0.8, colsample_bytree=0.8, eval_metric='mlogloss', random_state=42)

# solo features base (baseline ya conocido: ~0.46)
X_base = df_sup[base_feature_cols].values
f1_base = cross_val_score(xgb, X_base, y, cv=cv, scoring='f1_weighted').mean()

# features base + n-gramas
X_combined = df_sup[base_feature_cols + ngram_cols].values
f1_combined = cross_val_score(xgb, X_combined, y, cv=cv, scoring='f1_weighted').mean()

print(f"F1 solo features base:        {f1_base:.3f}")
print(f"F1 base + n-gramas:           {f1_combined:.3f}")
print(f"Diferencia:                   {f1_combined - f1_base:+.3f}")

---
## 11. Validating the Santa Valley Match — KH0323–KH0328
Reproducing Medrano & Urton (2018) using the OKR database directly.

In [ ]:
# Buscar por museo (Temple Radicati, Lima) y procedencia "Santa"
search1 = pd.read_sql("""
    SELECT KHIPU_ID, PROVENANCE, REGION, MUSEUM_NAME, NICKNAME
    FROM khipu_main
    WHERE MUSEUM_NAME LIKE '%Radicati%' 
       OR MUSEUM_NAME LIKE '%Temple%'
       OR PROVENANCE LIKE '%Santa%'
""", conn)
print("Búsqueda por museo/procedencia Santa:")
display(search1)

# Buscar por rango cercano de KHIPU_ID por si el offset es distinto
search2 = pd.read_sql("""
    SELECT KHIPU_ID, PROVENANCE, REGION, MUSEUM_NAME, NICKNAME
    FROM khipu_main
    WHERE KHIPU_ID BETWEEN 300 AND 350
    ORDER BY KHIPU_ID
""", conn)
print("\nKHIPU_ID entre 300-350:")
display(search2)

# Buscar columnas que puedan tener el número UR (Urton) o KH original
print("\nColumnas disponibles en khipu_main:")
print(khipu_main.columns.tolist() if 'khipu_main' in dir() else pd.read_sql("PRAGMA table_info(khipu_main)", conn))

In [ ]:
# Buscar por INVESTIGATOR_NUM (probablemente formato "UR087")
search3 = pd.read_sql("""
    SELECT KHIPU_ID, PROVENANCE, REGION, MUSEUM_NAME, NICKNAME, 
           INVESTIGATOR_NUM, OKR_NUM, MUSEUM_NUM
    FROM khipu_main
    WHERE INVESTIGATOR_NUM LIKE '%UR087%'
       OR INVESTIGATOR_NUM LIKE '%UR088%'
       OR INVESTIGATOR_NUM LIKE '%UR089%'
       OR INVESTIGATOR_NUM LIKE '%UR090%'
       OR INVESTIGATOR_NUM LIKE '%UR091%'
       OR INVESTIGATOR_NUM LIKE '%UR092%'
       OR OKR_NUM LIKE '%323%'
       OR OKR_NUM LIKE '%KH0323%'
""", conn)
print("Búsqueda por INVESTIGATOR_NUM / OKR_NUM:")
display(search3)

# Ver ejemplos generales de qué formato tienen estas columnas
sample = pd.read_sql("""
    SELECT KHIPU_ID, INVESTIGATOR_NUM, OKR_NUM, MUSEUM_NUM
    FROM khipu_main
    LIMIT 20
""", conn)
print("\nEjemplos de formato (primeras 20 filas):")
display(sample)

In [ ]:
santa_real_ids = [1000270, 1000271, 1000272, 1000273, 1000274, 1000275]

santa_main = pd.read_sql(f"""
    SELECT KHIPU_ID, PROVENANCE, MUSEUM_NAME, INVESTIGATOR_NUM, OKR_NUM, MUSEUM_NUM
    FROM khipu_main
    WHERE KHIPU_ID IN ({','.join(map(str, santa_real_ids))})
    ORDER BY OKR_NUM
""", conn)
print("Santa Valley khipus confirmados:")
display(santa_main)

santa_cords = pd.read_sql(f"""
    SELECT KHIPU_ID, CORD_ID, CORD_LEVEL, TWIST, ATTACHMENT_TYPE, ATTACH_POS
    FROM cord
    WHERE KHIPU_ID IN ({','.join(map(str, santa_real_ids))})
    ORDER BY KHIPU_ID, ATTACH_POS
""", conn)
print(f"\nTotal cordeles: {len(santa_cords)}")
print("\nCordeles por khipu:")
print(santa_cords.groupby('KHIPU_ID').size())

print("\nATTACHMENT_TYPE:")
print(santa_cords['ATTACHMENT_TYPE'].value_counts())

In [ ]:
# proporción R/V a nivel cordel
total_rv = santa_cords[santa_cords['ATTACHMENT_TYPE'].isin(['R','V'])].shape[0]
r_count = (santa_cords['ATTACHMENT_TYPE'] == 'R').sum()
v_count = (santa_cords['ATTACHMENT_TYPE'] == 'V').sum()

print(f"Recto (R): {r_count} ({r_count/total_rv*100:.1f}%)")
print(f"Verso (V): {v_count} ({v_count/total_rv*100:.1f}%)")
print(f"Total R+V: {total_rv}")
print(f"U (sin clasificar): {(santa_cords['ATTACHMENT_TYPE']=='U').sum()}")

# por khipu individual
print("\nR/V por khipu:")
rv_by_khipu = santa_cords[santa_cords['ATTACHMENT_TYPE'].isin(['R','V'])].groupby(
    ['KHIPU_ID', 'ATTACHMENT_TYPE']).size().unstack(fill_value=0)
rv_by_khipu = rv_by_khipu.merge(santa_main[['KHIPU_ID','OKR_NUM']], on='KHIPU_ID')
display(rv_by_khipu)

# el paper compara a nivel de "grupos de 6 cordeles" - veamos CORD_LEVEL
print("\nDistribución de CORD_LEVEL:")
print(santa_cords['CORD_LEVEL'].value_counts())

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════╗
║   VALIDATION: Santa Valley Khipus (Medrano & Urton 2018)    ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  6 khipus identified in OKR: KH0323-KH0328 (UR087-UR092)    ║
║  Museum: Museo Temple Radicati, Lima                         ║
║                                                              ║
║  RECTO/VERSO REPLICATION                                     ║
║  · This study (cord-level): R=49.0%, V=51.0%                 ║
║  · Medrano & Urton (group-level): R=47%, V=53%                ║
║  · Independent computational replication: CONSISTENT         ║
║                                                              ║
║  MOIETY STRUCTURE CONFIRMED                                  ║
║  · 5 of 6 khipus are purely R or purely V                    ║
║  · KH0326 is the sole mixed khipu (40R/69V)                  ║
║  · Matches original finding exactly                          ║
║                                                              ║
║  SIGNIFICANCE                                                ║
║  · First independent computational verification using only  ║
║    the public OKR database (no physical object access)       ║
║  · Validates OKR data quality for this specimen set          ║
║  · Demonstrates reproducibility of khipu decipherment claims ║
╚══════════════════════════════════════════════════════════════╝
""")

---
## Summary & Limitations

This notebook reproduces all results reported in the accompanying paper:

| Result | Value |
|---|---|
| Clusters (UMAP+HDBSCAN) | 3, silhouette = 0.769 |
| Classification F1 (Inka Late Horizon, tuned) | 0.86 |
| Top SHAP feature | cord twist direction |
| Santa Valley R/V replication | 49.0% / 51.0% (paper: ~47%/53%) |
| Mixed khipu identified | KH0326 (matches original study) |
| Sequence n-grams | No improvement (ΔF1 ≈ −0.006) |

**Limitations:** the labeled subset for region classification is small
(135/619 specimens), capping classifier performance. The corpus carries
documented collection and recording biases tied to its colonial-era
assembly (see Cluster 2 analysis above). The Santa Valley replication
confirms the *structural* (recto/verso) pattern only — full numerical
matching to the 367-peso tribute total requires positional knot-value
decoding, left for future work.

**Ethical note:** khipus are Andean cultural patrimony, not neutral data.
This analysis is offered as a tool for understanding and eventual community
stewardship, not extraction. See the paper's Ethical Considerations section
for further discussion.

**Citation:**
Contreras, M. (2026). Structural Pattern Mining in Inka Khipus: Unsupervised
Clustering, Provenance Classification, and a Computational Validation of the
Santa Valley Match. arXiv preprint.

📓 Code: github.com/mcontrerasmalpar-pixel/khipu-ml  
🗄️ Dataset: Open Khipu Repository, DOI 10.5281/zenodo.18025748